# Week 6 - Q4 follow-up: Why is most of the outstanding money over a year old?
**Question.** Week 3 found Rs 5.80B outstanding on reliable invoices, 86% of it more than 365 days past due, and no customer type or region that stands out. Week 6 asks *why* the money is so old. Whether it will be collected is not tested, because the data has no collection records.

**Approach.** The unit of analysis is the **invoice** (17,640 invoices with a unique `invoice_id`, so their payments can be attributed). The 393 repeated-ID invoices (open issue UI-01) are set aside, as in Week 3. Five steps: (1) outstanding by due year, (2) how late any payment ever arrives, (3) what the open balances look like, (4) who the open balances belong to, (5) a labelled exposure range. Every cut is tested with an effect size and an interval, and the cuts are adjusted together (Holm).

**Definitions (same as Week 3).** Outstanding = `grand_total` (incl. GST) minus `total_paid_amount`. Days past due = as-of date minus `due_date`; the as-of date is 2025-03-14, the latest due date in the data, not today. **Limits:** the data holds no collections, write-off, dispute or credit-note records, so it can show *what happened to invoices*, not *what the business did about them*. The exposure range is a scenario, not a forecast.

In [1]:
import pandas as pd, numpy as np, os
from pathlib import Path
from scipy import stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
SURF,INK,INK2,GRID,BLUE,ORANGE,AQUA,MUTED="#fcfcfb","#0b0b0b","#52514e","#e4e3df","#2a78d6","#eb6834","#1baf7a","#b9b8b2"
def find_dir(env_var, marker, rels):
    """Environment variable first, then this folder and every parent folder, using the repository layout."""
    if os.environ.get(env_var) and (Path(os.environ[env_var]) / marker).exists():
        return Path(os.environ[env_var])
    for base in [Path.cwd(), *Path.cwd().parents]:
        for rel in rels:
            if (base / rel / marker).exists():
                return base / rel
    return None
DATA_DIR = find_dir("CADETX_MERGED", "invoices_with_payments.csv", ["merged", "data/merged", "week-02/reports/week_02/merged", "reports/week_02/merged"])
assert DATA_DIR is not None, "Could not find the merged datasets. Run from inside the repository, put a merged/ folder next to the notebook, or set the CADETX_MERGED environment variable to the folder."
OUT=Path("q4_outputs"); OUT.mkdir(exist_ok=True); CH=Path("q4_charts"); CH.mkdir(exist_ok=True)
SEED=42; NPERM=5000; NBOOT=5000
rng=np.random.default_rng(SEED)
checks=[]
def check(name,expected,actual,ok):
    checks.append(dict(Check_Item=name,Expected=expected,Actual=actual,Status="PASS" if ok else "FAIL")); assert ok,f"{name}: expected {expected}, got {actual}"
def wilson(x,n,z=1.959964):
    p=x/n; d=1+z*z/n; c=(p+z*z/(2*n))/d; h=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d; return c-h,c+h
def holm(p):
    p=np.asarray(p,float); m=len(p); o=np.argsort(p); adj=np.empty(m); run=0
    for r,i in enumerate(o): run=max(run,(m-r)*p[i]); adj[i]=min(1,run)
    return adj
def cramers_v(ct):
    chi=stats.chi2_contingency(ct,correction=False)[0]; return float(np.sqrt(chi/(ct.values.sum()*(min(ct.shape)-1))))
def ca_trend(x,n,t):
    x,n,t=map(np.asarray,(x,n,t)); N=n.sum(); p=x.sum()/N
    T=(t*(x-n*p)).sum(); V=p*(1-p)*((n*t*t).sum()-((n*t).sum())**2/N); z=T/np.sqrt(V); return z,2*stats.norm.sf(abs(z))
def style(ax):
    ax.set_facecolor(SURF); ax.figure.set_facecolor(SURF)
    for s in ("top","right"): ax.spines[s].set_visible(False)
    for s in ("left","bottom"): ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2); ax.yaxis.grid(True,color=GRID,lw=0.8); ax.set_axisbelow(True)
print("Merged datasets found in:",Path(*DATA_DIR.resolve().parts[-4:]))

Merged datasets found in: week-02/reports/week_02/merged


## Part 0. Load and integrity checks

In [2]:
d=pd.read_csv(DATA_DIR/"invoices_with_payments.csv",low_memory=False,parse_dates=["invoice_date","due_date","first_payment_date","last_payment_date","as_of_date"])
check("invoices",18033,len(d),len(d)==18033)
rel_mask=d.payment_total_reliable.astype(str).str.lower().isin(["true","1"])
r=d[rel_mask].copy(); x=d[~rel_mask]
check("reliable invoices (unique invoice_id)",17640,len(r),len(r)==17640)
check("repeated-ID invoices set aside (UI-01)",393,len(x),len(x)==393)
check("repeated-ID invoices are worth Rs 654M incl. GST",654,round(x.grand_total.sum()/1e6),round(x.grand_total.sum()/1e6)==654)
check("invoice_id is unique among the reliable invoices",0,int(r.invoice_id.duplicated().sum()),r.invoice_id.duplicated().sum()==0)
AS_OF=pd.Timestamp("2025-03-14")
check("one as-of date, 2025-03-14, equal to the latest due date",str(AS_OF.date()),sorted(d.as_of_date.dt.date.astype(str).unique()),list(d.as_of_date.unique())==[np.datetime64(AS_OF)] and d.due_date.max()==AS_OF)
r["out"]=r.grand_total-r.total_paid_amount
check("outstanding equals grand_total minus total paid (max gap under Rs 1)",0,round(float((r["out"]-r.outstanding_amount).abs().max()),4),(r["out"]-r.outstanding_amount).abs().max()<1)
check("no outstanding amount is below minus one paisa (75 paid invoices are overpaid by exactly 0.01 and 92 are short by exactly 0.01; both are rounding differences)",0,int((r.outstanding_amount<-0.015).sum()),(r.outstanding_amount<-0.015).sum()==0)
r["open"]=(r.outstanding_amount>0.015).astype(int)
check("an invoice is open exactly when its status is not Paid",0,int(((r.payment_status!="Paid").astype(int)!=r.open).sum()),((r.payment_status!="Paid").astype(int)!=r.open).sum()==0)
r["dsd"]=(AS_OF-r.due_date).dt.days           # days since due at the as-of date (negative = not yet due)
r["due_year"]=r.due_date.dt.year
tot_out=r.outstanding_amount.sum(); tot_inv=r.grand_total.sum()
check("outstanding on reliable invoices is Rs 5.80B (Week 3)",5.80,round(tot_out/1e9,2),round(tot_out/1e9,2)==5.80)
od=r[(r.open==1)&(r.dsd>0)]
check("overdue invoices 5,375 (Week 3)",5375,len(od),len(od)==5375)
check("days_overdue column agrees with as-of minus due date on every overdue invoice",0,int((od.days_overdue-od.dsd).abs().max()),(od.days_overdue-od.dsd).abs().max()==0)
o365=od[od.dsd>365]
check("more than 365 days overdue: 4,595 invoices",4595,len(o365),len(o365)==4595)
check("more than 365 days overdue: Rs 4.99B",4.99,round(o365.outstanding_amount.sum()/1e9,2),round(o365.outstanding_amount.sum()/1e9,2)==4.99)
share365=o365.outstanding_amount.sum()/tot_out
check("86% of outstanding is more than 365 days overdue (Week 3)",86,round(share365*100),round(share365*100)==86)
check("only Delivered orders carry invoices",["Delivered"],sorted(r.order_status.unique()),set(r.order_status)=={"Delivered"})
print(f"{len(r):,} reliable invoices worth Rs {tot_inv/1e9:.2f}B incl. GST; outstanding Rs {tot_out/1e9:.2f}B on {int(r.open.sum()):,} open invoices; {share365*100:.1f}% of it more than 365 days past due")

17,640 reliable invoices worth Rs 28.60B incl. GST; outstanding Rs 5.80B on 5,376 open invoices; 85.9% of it more than 365 days past due


## Part 1. Outstanding by due year: is the backlog growing, or is it simply old?
If unpaid invoices slowly got collected, the open share would be *lower* for older due years. If a steady slice is never paid, the open share would be about the same in every due year, and the age mix of what is outstanding would just mirror how long ago the invoices fell due. The six full due years (2019 to 2024) are tested; 2025 holds only invoices due up to 14 March.

In [3]:
cy=r.groupby("due_year").agg(invoices=("open","size"),open_invoices=("open","sum"),invoiced=("grand_total","sum"),outstanding=("outstanding_amount","sum"))
cy["open_invoice_share_pct"]=cy.open_invoices/cy.invoices*100
cy["open_value_share_pct"]=cy.outstanding/cy.invoiced*100
ci=[wilson(a,b) for a,b in zip(cy.open_invoices,cy.invoices)]
cy["ci_low_pct"]=[c[0]*100 for c in ci]; cy["ci_high_pct"]=[c[1]*100 for c in ci]
full=cy.loc[2019:2024]
chi_y,p_y,_,_=stats.chi2_contingency(np.c_[full.open_invoices,full.invoices-full.open_invoices],correction=False)
V_y=float(np.sqrt(chi_y/(full.invoices.sum())))
z_t,p_t=ca_trend(full.open_invoices,full.invoices,np.arange(6))
sl=np.polyfit(np.arange(2019,2025),full.open_value_share_pct,1)[0]
# bootstrap the 2024 minus 2019 difference in open VALUE share
def vs(g,idx): return g.outstanding_amount.values[idx].sum()/g.grand_total.values[idx].sum()
g19,g24=r[r.due_year==2019],r[r.due_year==2024]
b=np.array([vs(g24,rng.integers(0,len(g24),len(g24)))-vs(g19,rng.integers(0,len(g19),len(g19))) for _ in range(NBOOT)])*100
diff=(vs(g24,np.arange(len(g24)))-vs(g19,np.arange(len(g19))))*100; dlo,dhi=np.percentile(b,[2.5,97.5])
print(cy.round(2).to_string())
print(f"\nOpen share equal across the six full years? chi-square p = {p_y:.3f} (Cramer's V {V_y:.3f})")
print(f"Downward trend (would mean older invoices get collected)? Cochran-Armitage z = {z_t:.2f}, p = {p_t:.3f}; fitted slope {sl:+.2f} points of value a year")
print(f"2024 minus 2019 open value share: {diff:+.2f} points, 95% bootstrap interval {dlo:+.2f} to {dhi:+.2f}")
cy.round(3).to_csv(OUT/"q4_open_by_due_year.csv")
pd.DataFrame([dict(test="open share equal across 2019-2024 (chi-square)",statistic=chi_y,p=p_y,effect=V_y),dict(test="trend across 2019-2024 (Cochran-Armitage)",statistic=z_t,p=p_t,effect=sl),dict(test="2024 minus 2019 open value share (bootstrap)",statistic=diff,p=np.nan,effect=f"{dlo:.2f} to {dhi:.2f}")]).round(4).to_csv(OUT/"q4_due_year_tests.csv",index=False)
check("six full due years each hold 2,500 to 3,000 invoices",True,[int(v) for v in full.invoices],bool(full.invoices.between(2500,3000).all()))
check("open value share sits between 19% and 22% in every full year",(19,22),(round(full.open_value_share_pct.min(),1),round(full.open_value_share_pct.max(),1)),bool(full.open_value_share_pct.between(19,22).all()))
check("open share is not significantly different across the six years (p > 0.05)",">0.05",round(p_y,3),p_y>0.05)
check("no downward trend with age: older invoices are not collected more (trend p > 0.05)",">0.05",round(p_t,3),p_t>0.05)
check("2024 vs 2019 difference interval includes zero",True,(round(dlo,2),round(dhi,2)),dlo<0<dhi)

          invoices  open_invoices      invoiced   outstanding  open_invoice_share_pct  open_value_share_pct  ci_low_pct  ci_high_pct
due_year                                                                                                                            
2019          2596            773  4.277943e+09  8.376433e+08                   29.78                 19.58       28.05        31.56
2020          2951            926  4.834369e+09  1.007853e+09                   31.38                 20.85       29.73        33.08
2021          2961            908  4.773585e+09  9.625247e+08                   30.67                 20.16       29.03        32.35
2022          2940            904  4.821026e+09  1.017815e+09                   30.75                 21.11       29.11        32.44
2023          2944            897  4.794791e+09  9.581042e+08                   30.47                 19.98       28.83        32.16
2024          2902            870  4.548355e+09  9.261047e+08        

In [4]:
# The age mix of what is outstanding mirrors the age mix of what was invoiced.
bins=[-10**6,0,365,10**6]; labs=["Not yet due","1-365 days past due","More than 365 days past due"]
r["age_band"]=pd.cut(r.dsd,bins=bins,labels=labs)
mix=r.groupby("age_band",observed=True).agg(invoiced=("grand_total","sum"),outstanding=("outstanding_amount","sum"))
mix["invoiced_share_pct"]=mix.invoiced/mix.invoiced.sum()*100; mix["outstanding_share_pct"]=mix.outstanding/mix.outstanding.sum()*100
over=r.dsd>365
def gap(idx): 
    s=r.iloc[idx]; return (s.loc[s.dsd>365,"outstanding_amount"].sum()/s.outstanding_amount.sum()-s.loc[s.dsd>365,"grand_total"].sum()/s.grand_total.sum())*100
n=len(r); gb=np.array([gap(rng.integers(0,n,n)) for _ in range(1000)])
gap_obs=gap(np.arange(n)); glo,ghi=np.percentile(gb,[2.5,97.5])
V=mix.loc[labs[2],"invoiced_share_pct"]; O=mix.loc[labs[2],"outstanding_share_pct"]
print(mix.round(2).to_string())
print(f"\nShare of invoiced value that fell due more than 365 days before the as-of date: {V:.1f}%")
print(f"Share of outstanding value that is more than 365 days past due: {O:.1f}%")
print(f"Difference {gap_obs:+.2f} points, 95% bootstrap interval {glo:+.2f} to {ghi:+.2f} (1,000 resamples)")
mix.round(3).to_csv(OUT/"q4_age_mix_vs_invoiced.csv")
check("86% of outstanding is over a year old, against 85% of invoiced value that fell due over a year ago: the gap is under 3 points",True,(round(O,1),round(V,1)),abs(O-V)<3)
check("the interval for the gap between the two shares excludes a gap larger than 5 points",True,(round(glo,2),round(ghi,2)),max(abs(glo),abs(ghi))<5)

                                 invoiced   outstanding  invoiced_share_pct  outstanding_share_pct
age_band                                                                                          
Not yet due                  2.843445e+06  9.918952e+05                0.01                   0.02
1-365 days past due          4.201576e+09  8.145421e+08               14.69                  14.03
More than 365 days past due  2.439522e+10  4.988770e+09               85.30                  85.95

Share of invoiced value that fell due more than 365 days before the as-of date: 85.3%
Share of outstanding value that is more than 365 days past due: 85.9%
Difference +0.65 points, 95% bootstrap interval -0.57 to +1.69 (1,000 resamples)


## Part 2. How late does a payment ever arrive?
If late payers eventually settle, payments would keep arriving long after the due date. The data shows how far past the due date any payment was ever made, and what happened to invoices that went past that point.

In [5]:
pay=r[r.last_payment_date.notna()].copy()
pay["late_days"]=(pay.last_payment_date-pay.due_date).dt.days
q=pay.late_days.quantile([0,.25,.5,.75,.9,.99,1])
late_share=(pay.late_days>0).mean()
paid_inv=r[r.payment_status=="Paid"]
paid_late=(paid_inv.last_payment_date-paid_inv.due_date).dt.days
MAXLATE=int(pay.late_days.max())
beyond_all=r[r.dsd>MAXLATE]; beyond=beyond_all[beyond_all.open==1]
print("Days from due date to LAST payment (negative = paid early), all invoices with a payment:")
print(q.round(0).to_string()); print(f"\nShare of invoices with a payment whose last payment came after the due date: {late_share*100:.1f}%")
print(f"Latest payment in the data: {MAXLATE} days after the due date")
print(f"Invoices now more than {MAXLATE} days past due: {len(beyond_all):,}; still open: {len(beyond):,} (Rs {beyond.outstanding_amount.sum()/1e9:.2f}B); settled in full: {len(beyond_all)-len(beyond):,}, all on or before {MAXLATE} days past due")
print(f"Fully paid invoices: {len(paid_inv):,}; latest full settlement came {int(paid_late.max())} days after the due date")
tw=pd.DataFrame({"days_from_due_to_last_payment":q.index,"value":q.values}); tw.to_csv(OUT/"q4_payment_lateness_quantiles.csv",index=False)
check("payments are made at most 75 days after the due date",75,MAXLATE,MAXLATE==75)
check("no fully paid invoice was settled later than 75 days past due",0,int((paid_late>75).sum()),(paid_late>75).sum()==0)
check("5,274 open invoices are more than 75 days past due",5274,len(beyond),len(beyond)==5274)
check("fully paid invoices were closed within 75 days of the due date (latest 45 days)","<=75",int(paid_late.max()),paid_late.max()<=75)
bal75=beyond.outstanding_amount.sum()
check("balances more than 75 days past due are Rs 5.71B, 98% of outstanding",(5.71,98),(round(bal75/1e9,2),round(bal75/tot_out*100)),(round(bal75/1e9,2),round(bal75/tot_out*100))==(5.71,98))

Days from due date to LAST payment (negative = paid early), all invoices with a payment:
0.00   -60.0
0.25   -20.0
0.50    -2.0
0.75    16.0
0.90    30.0
0.99    57.0
1.00    75.0

Share of invoices with a payment whose last payment came after the due date: 45.3%
Latest payment in the data: 75 days after the due date
Invoices now more than 75 days past due: 17,268; still open: 5,274 (Rs 5.71B); settled in full: 11,994, all on or before 75 days past due
Fully paid invoices: 12,264; latest full settlement came 45 days after the due date


In [6]:
# Among invoices whose payment window has closed (due more than 75 days ago), is the open share lower for older invoices?
w=r[r.dsd>MAXLATE].copy()
w["band"]=pd.cut(w.dsd,[75,180,365,730,10**6],labels=["76-180","181-365","366-730","More than 730"])
wb=w.groupby("band",observed=True).agg(invoices=("open","size"),open_invoices=("open","sum"),invoiced=("grand_total","sum"),outstanding=("outstanding_amount","sum"))
wb["open_invoice_share_pct"]=wb.open_invoices/wb.invoices*100; wb["open_value_share_pct"]=wb.outstanding/wb.invoiced*100
wb.round(3).to_csv(OUT/"q4_window_closed_by_age.csv")
print(wb.round(2).to_string())
print("\nOnce the payment window has closed, the share of invoices still open is about the same at every age: later settlement would have lowered it for older invoices.")

               invoices  open_invoices      invoiced   outstanding  open_invoice_share_pct  open_value_share_pct
band                                                                                                            
76-180              848            259  1.330004e+09  2.751874e+08                   30.54                 20.69
181-365            1452            420  2.292476e+09  4.446604e+08                   28.93                 19.40
366-730            2961            896  4.755549e+09  9.498719e+08                   30.26                 19.97
More than 730     12007           3699  1.963967e+10  4.038898e+09                   30.81                 20.57

Once the payment window has closed, the share of invoices still open is about the same at every age: later settlement would have lowered it for older invoices.


## Part 3. What do the open balances look like?
Open balances split into invoices with no payment at all and invoices paid in part.

In [7]:
ops=r[r.open==1]
comp=ops.groupby("payment_status").agg(invoices=("open","size"),outstanding=("outstanding_amount","sum"),invoiced=("grand_total","sum"))
comp["share_of_outstanding_pct"]=comp.outstanding/tot_out*100
pp=r[r.payment_status=="Partially Paid"].copy(); pp["paid_frac"]=pp.total_paid_amount/pp.grand_total
ks=stats.kstest(pp.paid_frac,"uniform",args=(0.2,0.6))
rho_a,p_a=stats.spearmanr(pp.paid_frac,pp.dsd); rho_t,p_t2=stats.spearmanr(pp.paid_frac,pp.due_year)
print(comp.round(2).to_string())
print(f"\nPartly paid invoices ({len(pp):,}): paid fraction mean {pp.paid_frac.mean():.3f}, range {pp.paid_frac.min():.2f} to {pp.paid_frac.max():.2f}")
print(f"Is the paid fraction spread evenly between 20% and 80%? Kolmogorov-Smirnov D = {ks.statistic:.3f}, p = {ks.pvalue:.3f}")
print(f"Does the paid fraction depend on how old the invoice is? Spearman rho = {rho_a:+.3f} (p = {p_a:.2f}); on due year rho = {rho_t:+.3f} (p = {p_t2:.2f})")
comp.round(3).to_csv(OUT/"q4_open_composition.csv"); pp[["invoice_id","grand_total","total_paid_amount","paid_frac","dsd"]].round(4).describe().to_csv(OUT/"q4_partial_payment_summary.csv")
check("unpaid invoices: 1,798 worth Rs 2.92B outstanding",(1798,2.92),(int(comp.loc["Unpaid","invoices"]),round(comp.loc["Unpaid","outstanding"]/1e9,2)),(int(comp.loc["Unpaid","invoices"]),round(comp.loc["Unpaid","outstanding"]/1e9,2))==(1798,2.92))
check("partly paid invoices: 3,578 worth Rs 2.89B outstanding",(3578,2.89),(int(comp.loc["Partially Paid","invoices"]),round(comp.loc["Partially Paid","outstanding"]/1e9,2)),(int(comp.loc["Partially Paid","invoices"]),round(comp.loc["Partially Paid","outstanding"]/1e9,2))==(3578,2.89))
check("open invoices are only Unpaid or Partially Paid",["Partially Paid","Unpaid"],sorted(ops.payment_status.unique()),set(ops.payment_status)=={"Unpaid","Partially Paid"})
check("partly paid fraction stays between 20% and 80%",(0.2,0.8),(round(pp.paid_frac.min(),2),round(pp.paid_frac.max(),2)),pp.paid_frac.min()>=0.199 and pp.paid_frac.max()<=0.801)
check("paid fraction is unrelated to invoice age (Spearman |rho| < 0.05)","<0.05",round(abs(rho_a),3),abs(rho_a)<0.05)

                invoices   outstanding      invoiced  share_of_outstanding_pct
payment_status                                                                
Partially Paid      3578  2.885861e+09  5.826085e+09                     49.72
Unpaid              1798  2.918443e+09  2.918443e+09                     50.28

Partly paid invoices (3,578): paid fraction mean 0.499, range 0.20 to 0.80
Is the paid fraction spread evenly between 20% and 80%? Kolmogorov-Smirnov D = 0.011, p = 0.757
Does the paid fraction depend on how old the invoice is? Spearman rho = +0.003 (p = 0.84); on due year rho = +0.002 (p = 0.90)


## Part 4. Who do the open balances belong to?
Each cut is tested on whether an invoice is open (chi-square, Cramer's V), with the open share of value shown for each level. All cuts are adjusted together (Holm). Payment terms get a closer look, because an `Advance` invoice should be paid before delivery.

In [8]:
cuts=["payment_terms","customer_type","customer_region","industry_segment","sales_channel","region"]
rows=[];seg=[]
for c in cuts:
    ct=pd.crosstab(r[c],r.open)
    chi,p,_,_=stats.chi2_contingency(ct,correction=False); v=cramers_v(ct)
    g=r.groupby(c).agg(n=("open","size"),o=("open","sum"),inv=("grand_total","sum"),out=("outstanding_amount","sum"))
    g["open_invoice_share_pct"]=g.o/g.n*100; g["open_value_share_pct"]=g.out/g.inv*100
    for lv,row in g.iterrows():
        lo,hi=wilson(row.o,row.n); seg.append(dict(cut=c,level=lv,invoices=int(row.n),open_invoice_share_pct=row.open_invoice_share_pct,ci_low_pct=lo*100,ci_high_pct=hi*100,open_value_share_pct=row.open_value_share_pct))
    rows.append(dict(cut=c,levels=len(g),chi2=chi,p_raw=p,cramers_v=v,open_value_share_min_pct=g.open_value_share_pct.min(),open_value_share_max_pct=g.open_value_share_pct.max()))
T=pd.DataFrame(rows); T["p_holm"]=holm(T.p_raw)
S=pd.DataFrame(seg)
print(T.round(4).to_string(index=False)); print()
print(S[S.cut=="payment_terms"].round(2).to_string(index=False))
T.round(5).to_csv(OUT/"q4_segment_tests.csv",index=False); S.round(3).to_csv(OUT/"q4_segment_rates.csv",index=False)
adv=r.payment_terms=="Advance"
chi_a,p_adv,_,_=stats.chi2_contingency(pd.crosstab(adv,r.open),correction=False)
print(f"\nAdvance invoices open: {r[adv].open.mean()*100:.1f}%; other terms open: {r[~adv].open.mean()*100:.1f}% (chi-square p = {p_adv:.3f})")
check("six cuts tested together",6,len(T),len(T)==6)
check("no cut is significant after the Holm adjustment (smallest adjusted p > 0.05)",">0.05",round(float(T.p_holm.min()),3),T.p_holm.min()>0.05)
check("every Cramer's V is below 0.03 (negligible effect)","<0.03",round(float(T.cramers_v.max()),4),T.cramers_v.max()<0.03)
check("open value share is within 17% to 23% at every level of every cut",(17,23),(round(float(S.open_value_share_pct.min()),1),round(float(S.open_value_share_pct.max()),1)),S.open_value_share_pct.between(17,23).all())

             cut  levels   chi2  p_raw  cramers_v  open_value_share_min_pct  open_value_share_max_pct  p_holm
   payment_terms       5 5.3287 0.2552     0.0174                   19.3163                   21.1596  1.0000
   customer_type       5 4.7509 0.3138     0.0164                   19.6445                   21.4169  1.0000
 customer_region       6 4.0977 0.5354     0.0152                   18.5751                   20.9082  1.0000
industry_segment       6 2.4287 0.7872     0.0117                   19.0065                   22.1250  1.0000
   sales_channel       4 1.2415 0.7431     0.0084                   19.8123                   20.8069  1.0000
          region       4 7.5541 0.0562     0.0207                   19.7058                   20.7996  0.3371

          cut   level  invoices  open_invoice_share_pct  ci_low_pct  ci_high_pct  open_value_share_pct
payment_terms Advance      3661                   29.25       27.80        30.75                 19.32
payment_terms  Net 15  


Advance invoices open: 29.3%; other terms open: 30.8% (chi-square p = 0.071)


In [9]:
# Customer clustering: is the unpaid money concentrated in a few customers beyond chance?
# Null: each invoice keeps its value and customer, but WHICH invoices are open is shuffled across all invoices.
codes,uniq=pd.factorize(r.customer_id); K=len(uniq)
val_c=np.bincount(codes,weights=r.grand_total.values,minlength=K)
u=(r.outstanding_amount/r.grand_total).values; g=r.grand_total.values
def stat(u_):
    o=np.bincount(codes,weights=g*u_,minlength=K); s=o/val_c
    return s.std(ddof=0), np.sort(o)[-50:].sum()/o.sum(), s.max()
obs=stat(u); nulls=np.array([stat(u[rng.permutation(len(u))]) for _ in range(NPERM)])
pvals=[(1+(nulls[:,i]>=obs[i]).sum())/(1+NPERM) for i in range(3)]
cust_share=pd.Series(np.bincount(codes,weights=r.outstanding_amount.values,minlength=K)/val_c,index=uniq)
chi_c,p_c,_,_=stats.chi2_contingency(np.c_[r.groupby("customer_id").open.sum(),r.groupby("customer_id").open.count()-r.groupby("customer_id").open.sum()],correction=False)
top50=obs[1]*100; top50_exp=np.mean(nulls[:,1])*100
print(f"Customers: {K}; invoices per customer {r.groupby('customer_id').size().min()} to {r.groupby('customer_id').size().max()}")
print(f"SD of customer open value share: observed {obs[0]*100:.2f} points, chance mean {nulls[:,0].mean()*100:.2f} (95th pct {np.percentile(nulls[:,0],95)*100:.2f}); p = {pvals[0]:.3f}")
print(f"Share of outstanding held by the 50 customers owing most (10% of customers): observed {top50:.2f}%, chance mean {top50_exp:.2f}% (95th pct {np.percentile(nulls[:,1],95)*100:.2f}); p = {pvals[1]:.3f}")
print(f"Highest customer open value share: {obs[2]*100:.1f}% against chance maximum mean {nulls[:,2].mean()*100:.1f}% (p = {pvals[2]:.3f})")
print(f"Chi-square on open invoices across the {K} customers: p = {p_c:.3f}")
cust=pd.DataFrame({"customer_id":uniq,"invoices":np.bincount(codes),"invoiced":val_c,"open_value_share_pct":cust_share.values*100,"outstanding":np.bincount(codes,weights=r.outstanding_amount.values,minlength=K)}).sort_values("outstanding",ascending=False)
cust.round(3).to_csv(OUT/"q4_customer_open_share.csv",index=False)
pd.DataFrame([dict(statistic="SD of customer open value share (pts)",observed=obs[0]*100,chance_mean=nulls[:,0].mean()*100,chance_95th=np.percentile(nulls[:,0],95)*100,p=pvals[0]),
              dict(statistic="share of outstanding held by top 50 customers (%)",observed=top50,chance_mean=top50_exp,chance_95th=np.percentile(nulls[:,1],95)*100,p=pvals[1]),
              dict(statistic="highest customer open value share (%)",observed=obs[2]*100,chance_mean=nulls[:,2].mean()*100,chance_95th=np.percentile(nulls[:,2],95)*100,p=pvals[2])]).round(4).to_csv(OUT/"q4_customer_clustering_test.csv",index=False)
check("500 customers carry the reliable invoices",500,K,K==500)
check("customer clustering of unpaid value is not above chance (spread test p > 0.05)",">0.05",round(pvals[0],3),pvals[0]>0.05)
check("the 50 customers owing most hold no more than chance (p > 0.05)",">0.05",round(pvals[1],3),pvals[1]>0.05)
check("chi-square across customers is not significant (p > 0.05)",">0.05",round(p_c,3),p_c>0.05)

Customers: 500; invoices per customer 21 to 55
SD of customer open value share: observed 7.27 points, chance mean 7.61 (95th pct 8.02); p = 0.916
Share of outstanding held by the 50 customers owing most (10% of customers): observed 18.38%, chance mean 18.66% (95th pct 19.32); p = 0.748
Highest customer open value share: 44.3% against chance maximum mean 47.6% (p = 0.800)
Chi-square on open invoices across the 500 customers: p = 0.361


## Part 5. Exposure range (scenario, not forecast)
No invoice in the data was ever settled more than 75 days past its due date, so the balances now more than 75 days past due have no precedent of closing. The table shows what part of the invoiced value would be at stake if 25%, 50%, 75% or all of those balances were never collected. It is a range for planning, not a prediction: the data holds no collection or write-off records.

In [10]:
ex=[]
for pct in (25,50,75,100):
    lost=bal75*pct/100
    ex.append(dict(never_collected_share_of_balances_past_75_days_pct=pct,exposure_rs_b=lost/1e9,share_of_outstanding_pct=lost/tot_out*100,share_of_invoiced_value_pct=lost/tot_inv*100))
EX=pd.DataFrame(ex); EX.round(3).to_csv(OUT/"q4_exposure_range.csv",index=False)
print(EX.round(2).to_string(index=False))
print(f"\nBase: Rs {bal75/1e9:.2f}B of balances more than 75 days past due, out of Rs {tot_inv/1e9:.2f}B invoiced (incl. GST, reliable invoices)")
check("exposure rows are 25, 50, 75 and 100% of the Rs 5.71B past-75-day balances",(1.43,2.85,4.28,5.71),tuple(round(v,2) for v in EX.exposure_rs_b),tuple(round(v,2) for v in EX.exposure_rs_b)==(1.43,2.85,4.28,5.71))
check("full exposure is 20.0% of invoiced value",20.0,round(EX.share_of_invoiced_value_pct.iloc[-1],1),round(EX.share_of_invoiced_value_pct.iloc[-1],1)==20.0)

 never_collected_share_of_balances_past_75_days_pct  exposure_rs_b  share_of_outstanding_pct  share_of_invoiced_value_pct
                                                 25           1.43                     24.59                         4.99
                                                 50           2.85                     49.18                         9.98
                                                 75           4.28                     73.76                        14.97
                                                100           5.71                     98.35                        19.96

Base: Rs 5.71B of balances more than 75 days past due, out of Rs 28.60B invoiced (incl. GST, reliable invoices)


## Charts

In [11]:
fig,ax=plt.subplots(figsize=(8,4.4)); style(ax)
yrs=cy.index.astype(str); v=cy.open_value_share_pct.values
cols=[BLUE]*6+[MUTED]
ax.bar(yrs,v,color=cols,width=0.62)
for i,val in enumerate(v): ax.text(i,val+0.5,f"{val:.1f}%",ha="center",color=INK,fontsize=10)
ax.axhline(full.outstanding.sum()/full.invoiced.sum()*100,color=ORANGE,lw=1.6,ls="--"); ax.text(-0.4,23.8,"Dashed line: 2019-2024 average",color=ORANGE,ha="left",fontsize=10)
ax.set_ylim(0,26); ax.set_ylabel("Outstanding as % of invoiced value",color=INK2)
ax.set_title("The open share is about the same for every due year",loc="left",color=INK,fontsize=13,fontweight="bold")
ax.text(0,-0.18,"2025 (grey) holds only invoices due up to 14 March 2025.",transform=ax.transAxes,color=INK2,fontsize=9)
fig.tight_layout(); fig.savefig(CH/"q4_01_open_share_by_due_year.png",dpi=160); plt.close(fig)

In [12]:
fig,ax=plt.subplots(figsize=(8,4.4)); style(ax)
ax.hist(pay.late_days,bins=np.arange(-62,80,3),color=BLUE,edgecolor=SURF)
ax.axvline(75,color=ORANGE,lw=1.8); ax.text(73,ax.get_ylim()[1]*0.9,"latest payment:\n75 days after due",color=ORANGE,ha="right",fontsize=10)
ax.set_xlabel("Days from due date to last payment (negative = paid early)",color=INK2); ax.set_ylabel("Invoices",color=INK2)
ax.set_title("No payment arrives more than 75 days after the due date",loc="left",color=INK,fontsize=13,fontweight="bold")
ax.text(0,-0.2,f"{len(beyond):,} open invoices are now past that point; none has ever settled.",transform=ax.transAxes,color=INK2,fontsize=9)
fig.tight_layout(); fig.savefig(CH/"q4_02_payment_lateness.png",dpi=160); plt.close(fig)

In [13]:
fig,ax=plt.subplots(figsize=(8,4.4)); style(ax)
xs=np.arange(3); w=0.36
ax.bar(xs-w/2,mix.invoiced_share_pct,w,color=MUTED,label="Share of invoiced value"); ax.bar(xs+w/2,mix.outstanding_share_pct,w,color=BLUE,label="Share of outstanding value")
for i in range(3):
    ax.text(i-w/2,mix.invoiced_share_pct.iloc[i]+1.2,f"{mix.invoiced_share_pct.iloc[i]:.0f}%",ha="center",color=INK,fontsize=10); ax.text(i+w/2,mix.outstanding_share_pct.iloc[i]+1.2,f"{mix.outstanding_share_pct.iloc[i]:.0f}%",ha="center",color=INK,fontsize=10)
ax.set_xticks(xs); ax.set_xticklabels(["Not yet due","1-365 days\npast due","More than 365 days\npast due"],color=INK2); ax.set_ylim(0,100); ax.set_ylabel("Share of total (%)",color=INK2)
ax.legend(frameon=False,loc="upper left")
ax.set_title("The outstanding money is old because the invoices are old",loc="left",color=INK,fontsize=13,fontweight="bold")
fig.tight_layout(); fig.savefig(CH/"q4_03_age_mix.png",dpi=160); plt.close(fig)

In [14]:
fig,ax=plt.subplots(figsize=(8,4.4)); style(ax)
ax.hist(nulls[:,1]*100,bins=40,color=MUTED,edgecolor=SURF,label="Chance (5,000 shuffles)")
ax.set_ylim(0,ax.get_ylim()[1]*1.15); ax.axvline(top50,color=ORANGE,lw=2); ax.text(top50-0.03,ax.get_ylim()[1]*0.95,f"observed {top50:.1f}%",color=ORANGE,fontsize=10,ha="right")
ax.set_xlabel("Share of outstanding held by the 50 customers owing most (%)",color=INK2); ax.set_ylabel("Shuffles",color=INK2)
ax.set_title("Customers owe about what chance would give",loc="left",color=INK,fontsize=13,fontweight="bold")
fig.tight_layout(); fig.savefig(CH/"q4_04_customer_clustering.png",dpi=160); plt.close(fig)

## Findings

In [15]:
print("FINDINGS")
print(f"1. The outstanding money is old because the invoices are old. About {full.outstanding.sum()/full.invoiced.sum()*100:.0f}% of the value due in every full year (2019-2024) is still open, with no downward trend (p = {p_t:.2f}); {O:.0f}% of outstanding is over a year past due against {V:.0f}% of invoiced value that fell due over a year ago.")
print(f"2. No payment in the data arrives more than {MAXLATE} days after the due date, so an invoice still open beyond that point has no precedent of closing. {len(beyond):,} invoices are in that position (Rs {bal75/1e9:.2f}B, {bal75/tot_out*100:.0f}% of outstanding).")
print(f"3. Open balances are Rs {comp.loc['Unpaid','outstanding']/1e9:.2f}B on unpaid invoices and Rs {comp.loc['Partially Paid','outstanding']/1e9:.2f}B on partly paid ones; partly paid invoices were paid {pp.paid_frac.min()*100:.0f}% to {pp.paid_frac.max()*100:.0f}% with no link to age.")
print(f"4. Nobody stands out: no cut is significant after Holm (smallest adjusted p = {T.p_holm.min():.2f}, largest Cramer's V = {T.cramers_v.max():.3f}). Advance-terms invoices are open about as often as the rest ({r[adv].open.mean()*100:.1f}% against {r[~adv].open.mean()*100:.1f}%, p = {p_adv:.2f}), so payment terms make no practical difference; customer spread is within chance (p = {pvals[0]:.2f}).")
print(f"5. Exposure if 25% to 100% of the past-{MAXLATE}-day balances were never collected: Rs {EX.exposure_rs_b.iloc[0]:.2f}B to Rs {EX.exposure_rs_b.iloc[-1]:.2f}B, or {EX.share_of_invoiced_value_pct.iloc[0]:.1f}% to {EX.share_of_invoiced_value_pct.iloc[-1]:.1f}% of invoiced value.")
print("LIMIT: the data has no collection, dispute, credit-note or write-off records, so it cannot say whether the money WILL be collected; it shows that it has not been.")

FINDINGS
1. The outstanding money is old because the invoices are old. About 20% of the value due in every full year (2019-2024) is still open, with no downward trend (p = 0.77); 86% of outstanding is over a year past due against 85% of invoiced value that fell due over a year ago.
2. No payment in the data arrives more than 75 days after the due date, so an invoice still open beyond that point has no precedent of closing. 5,274 invoices are in that position (Rs 5.71B, 98% of outstanding).
3. Open balances are Rs 2.92B on unpaid invoices and Rs 2.89B on partly paid ones; partly paid invoices were paid 20% to 80% with no link to age.
4. Nobody stands out: no cut is significant after Holm (smallest adjusted p = 0.34, largest Cramer's V = 0.021). Advance-terms invoices are open about as often as the rest (29.3% against 30.8%, p = 0.07), so payment terms make no practical difference; customer spread is within chance (p = 0.92).
5. Exposure if 25% to 100% of the past-75-day balances were ne

## Validation report

In [16]:
rep=pd.DataFrame(checks); rep.insert(0,"Check_ID",range(1,len(rep)+1)); rep.to_csv("validation_report_week06_q4.csv",index=False)
print(rep.Status.value_counts().to_dict()); assert (rep.Status=="PASS").all()
print(f"{len(rep)} checks, all PASS")

{'PASS': 43}
43 checks, all PASS
